# Real-Time Driver Drowsiness Detection using YOLOv5

This notebook demonstrates the complete end-to-end pipeline for driver drowsiness detection using YOLOv5:
1. Environment setup & dependency installation
2. Pretrained model loading & hub verification
3. Image detection & visualization
4. Real-time webcam inference feed
5. Image collection & custom YOLO training setup (`awake` vs `drowsy`)
6. Custom weight inference & real-time drowsiness alert monitor

## Step 1: Install and Import Dependencies

In [ ]:
# Install core requirements (uncomment if running in fresh environment)
# !pip install torch torchvision torchaudio
# !git clone https://github.com/ultralytics/yolov5
# !cd yolov5 && pip install -r requirements.txt

import torch
from matplotlib import pyplot as plt
import numpy as np
import cv2
import os
import uuid
import time

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

## Step 2: Load YOLOv5 Pretrained Model

In [ ]:
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
model

## Step 3: Make Detections with Sample Images

In [ ]:
img = 'https://upload.wikimedia.org/wikipedia/commons/thumb/e/e4/Cars_in_traffic_in_Auckland%2C_New_Zealand_-_copyright-free_photo_released_to_public_domain.jpg/800px-Cars_in_traffic_in_Auckland%2C_New_Zealand_-_copyright-free_photo_released_to_public_domain.jpg'
results = model(img)
results.print()

%matplotlib inline 
plt.figure(figsize=(10, 8))
plt.imshow(np.squeeze(results.render()))
plt.axis('off')
plt.show()

## Step 4: Real-Time Detections via Webcam

In [ ]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Webcam feed unavailable or disconnected.")
        break
        
    # Make detections 
    results = model(frame)
    
    cv2.imshow('YOLO Real-Time Detection', np.squeeze(results.render()))
    
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

## Step 5: Data Collection & Custom Training Setup

In [ ]:
IMAGES_PATH = os.path.join('data', 'images') # /data/images
labels = ['awake', 'drowsy']
number_imgs = 5

os.makedirs(IMAGES_PATH, exist_ok=True)

cap = cv2.VideoCapture(0)
# Loop through labels
for label in labels:
    print(f'Collecting images for {label}. Get ready!')
    time.sleep(5)
    
    # Loop through image range
    for img_num in range(number_imgs):
        print(f'Collecting images for {label}, image number {img_num+1}/{number_imgs}')
        
        # Webcam feed
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame.")
            break
        
        # Naming out image path
        imgname = os.path.join(IMAGES_PATH, f"{label}.{str(uuid.uuid1())}.jpg")
        
        # Writes out image to file 
        cv2.imwrite(imgname, frame)
        
        # Render to the screen
        cv2.imshow('Image Collection', frame)
        
        # 2 second delay between captures
        time.sleep(2)
        
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# Print image paths verification
for label in labels:
    print(f'Verification for {label}:')
    for img_num in range(number_imgs):
        imgname = os.path.join(IMAGES_PATH, f"{label}.{str(uuid.uuid1())}.jpg")
        print("  ", imgname)

In [ ]:
# Labeling and Training Setup Commands
# !git clone https://github.com/tzutalin/labelImg
# !pip install pyqt5 lxml --upgrade
# !cd labelImg && pyrcc5 -o libs/resources.py resources.qrc

# Launch training command:
# !cd yolov5 && python train.py --img 320 --batch 16 --epochs 500 --data ../dataset.yml --weights yolov5s.pt --workers 2

## Step 6: Load Custom Trained Model & Real-Time Drowsiness Detection

In [ ]:
weights_path = os.path.join('yolov5', 'runs', 'train', 'exp15', 'weights', 'last.pt')

if os.path.exists(weights_path):
    model = torch.hub.load('ultralytics/yolov5', 'custom', path=weights_path, force_reload=True)
    print(f"✓ Successfully loaded custom weights from: {weights_path}")
else:
    print(f"ℹ Custom weights '{weights_path}' not found. Loading pretrained 'yolov5s' model.")
    model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

# Real-Time Detection Feed
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Webcam stream interrupted.")
        break
    
    # Make detections 
    results = model(frame)
    
    cv2.imshow('Real-Time Drowsiness Detection (YOLO)', np.squeeze(results.render()))
    
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()